In [2]:
import os
import glob
import sys
from pathlib import Path
import pandas as pd
import plotly.express as px
import seaborn as sns

cwd = Path.cwd().resolve()
src_dir = cwd / "src" if (cwd / "src").exists() else cwd.parent / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from utils.io import ensure_output_dir

RESULTS_DIRECTORY = '../results'
OUTPUT_DIR = ensure_output_dir(
    base_output_dir=RESULTS_DIRECTORY,
    lif_container_id='data_analysis_exploratory',
    results_type='tabular_summaries'
)

CONCATENATED_RESULTS_PATH = OUTPUT_DIR / 'concatenated_results.csv'
CONCATENATED_RESULTS_MEAN_PATH = OUTPUT_DIR / 'concatenated_results_mean.csv'

# Recursively find all .csv files in subdirectories under RESULTS_DIRECTORY
csv_file_list = glob.glob(os.path.join(RESULTS_DIRECTORY, '**', '*.csv'), recursive=True)

# Exclude generated summary files from inputs
csv_file_list = [f for f in csv_file_list if os.path.abspath(f) not in {
    str(CONCATENATED_RESULTS_PATH.resolve()),
    str(CONCATENATED_RESULTS_MEAN_PATH.resolve()),
}]

# Read all CSVs into a list of DataFrames
dfs = [pd.read_csv(f) for f in csv_file_list]

# Concatenate all DataFrames
concatenated_df = pd.concat(dfs, ignore_index=True)

# Save as concatenated_results.csv for further processing
concatenated_df.to_csv(CONCATENATED_RESULTS_PATH, index=False)

In [ ]:
concatenated_df.head(5)

In [ ]:
# Extract experiment fields from lif_container_id
experiment_fields = concatenated_df['lif_container_id'].str.split('_', expand=True)
concatenated_df['experiment_date'] = experiment_fields[0]
concatenated_df['experiment_name'] = experiment_fields[1]
concatenated_df['treatment'] = experiment_fields[2]
concatenated_df['experiment_id'] = experiment_fields[0] + '_' + experiment_fields[1]

# Extract genotype and replica_id from lif_image_name
image_fields = concatenated_df['lif_image_name'].str.split(' ', expand=True)
concatenated_df['genotype'] = image_fields[0]
concatenated_df['replica_id'] = image_fields[1]

# Reorder columns: keep existing ID columns once, then insert only derived fields
cols = list(concatenated_df.columns)
li_index = cols.index('lif_image_name')
front = cols[:li_index + 1]
derived = ['experiment_date', 'experiment_name', 'treatment', 'experiment_id', 'genotype', 'replica_id']
new_order = front + derived + [c for c in cols if c not in (front + derived)]
concatenated_df = concatenated_df[new_order]

# Save the updated DataFrame, overwriting the file
concatenated_df.to_csv(CONCATENATED_RESULTS_PATH, index=False)

concatenated_df.head(5)

In [ ]:
# Group by 'lif_container_id', 'lif_image_name', 'tissue_layer' and keep non-numeric (non-aggregated) columns
# Directly derived non-numeric columns to retain
non_numeric_columns = [
    'experiment_date', 'experiment_name', 'treatment', 'experiment_id', 'genotype', 'replica_id'
]

# Numeric columns for aggregation
numeric_columns = concatenated_df.select_dtypes(include='number').columns.tolist()

# Combine for aggregation: use 'mean' for numerics, 'first' for non-numerics
agg_dict = {col: 'mean' for col in numeric_columns}
agg_dict.update({col: 'first' for col in non_numeric_columns if col in concatenated_df.columns})

grouped_df = (
    concatenated_df
    .groupby(['lif_container_id', 'lif_image_name', 'tissue_layer'], as_index=False)
    .agg(agg_dict)
)

grouped_df.to_csv(CONCATENATED_RESULTS_MEAN_PATH, index=False)

grouped_df.head(20)

In [ ]:
grouped_df.columns

In [ ]:
columns_to_plot = ['area', 'FRET_ratio_sum_norm_per_image', 'FRET_ratio_mean_norm_per_image', 'FRET_ratio_sum', 'FRET_ratio_mean']

In [ ]:
for column in columns_to_plot:
    fig = px.scatter(
        grouped_df,
        x='tissue_layer',
        y=column,
        color='treatment',  # Color code by treatment
        hover_data=['lif_container_id', 'lif_image_name', 'area', 'treatment'],
        title=f"{column} by Tissue Layer (Colored by Treatment)",
    )
    fig.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import cm

# Define feature pairs to plot as (x, y) tuples for both dataframes
features_to_plot = [
    ("area", "edCitrine_CTRL_mean_int"),
    # Add or modify pairs as needed
]

# Define a consistent colormap for tissue_layer using viridis
# You can change "viridis" to "plasma", "magma", "cividis", "Set1", etc. to experiment.
hue_palette = "viridis"  # researcher can set this to any matplotlib or seaborn palette string, e.g., "plasma", "cividis", "tab10"

# Build a dictionary mapping tissue_layer categories to colors from the colormap
def get_tissue_layer_palette(df, col, palette_name):
    categories = sorted(df[col].dropna().unique())
    cmap = cm.get_cmap(palette_name, len(categories))
    colors = [cmap(i) for i in range(len(categories))]
    return dict(zip(categories, colors))

# Create a union of categories across both DataFrames for consistent mapping
all_tissue_layers = sorted(set(concatenated_df["tissue_layer"].dropna().unique())
                          | set(grouped_df["tissue_layer"].dropna().unique()))
cmap = cm.get_cmap(hue_palette, len(all_tissue_layers))
tissue_layer_palette = dict(zip(all_tissue_layers, [cmap(i) for i in range(len(all_tissue_layers))]))

# Plotting from the raw concatenated_df (per-nucleus or per-object)
for feature_pair in features_to_plot:
    x_axis, y_axis = feature_pair
    joint = sns.jointplot(
        data=concatenated_df,
        x=x_axis,
        y=y_axis,
        hue="tissue_layer",
        palette=tissue_layer_palette,
        kind="scatter"  # Use "reg" for regression line or "kde" for density
    )
    joint.figure.suptitle(f"Jointplot of {x_axis} vs {y_axis} (concatenated_df)", y=1.02)
    joint.figure.tight_layout()
    # joint.figure.show()  # Uncomment if running in interactive environment
    plt.show()  # More broadly compatible with most environments

# Plotting from the grouped_df (grouped/aggregated by image/layer)
for feature_pair in features_to_plot:
    x_axis, y_axis = feature_pair
    joint = sns.jointplot(
        data=grouped_df,
        x=x_axis,
        y=y_axis,
        hue="tissue_layer",
        palette=tissue_layer_palette,
        kind="scatter"
    )
    joint.figure.suptitle(f"Jointplot of {x_axis} vs {y_axis} (grouped_df)", y=1.02)
    joint.figure.tight_layout()
    # joint.figure.show()  # Uncomment if running in interactive environment
    plt.show()  # More broadly compatible with most environments

